# A free GPU for the song site

Open it with [this link](https://colab.research.google.com/github/soluzka/AI-driven_NVIDIA_song_designer/blob/main/deploy/colab/song_gpu.ipynb),
choose **Runtime → Change runtime type → T4 GPU**, and press **Run all**. Nothing
is installed on your own computer, nothing is rented, and no account or card is
needed.

What each cell does, so a slow one is recognisable rather than a spinner:

| Cell | Prints | Takes |
| --- | --- | --- |
| the card | `torch … cuda True` | seconds |
| packages | `packages in place` | 3–6 minutes |
| weights | download bars, then `weights exit 0` | 5–15 minutes |
| the service | `service: {...}` | 1–3 minutes (the model is loading) |
| the downloader | `downloads: {...}` | under a minute |
| the tunnel | `ADDRESS: https://…` | under a minute |

The last cell prints a line beginning `ADDRESS:`. Press **Connect my model** in
the site header and paste it. If you have the owner's join key, put it in the
`JOIN_KEY` cell above that and the card joins the pool instead.

Do not paste this file's text into a cell: a notebook is JSON and Colab runs
pasted text as Python, which is the `NameError: name 'null' is not defined`
people see. The link above opens it as a notebook. If that link will not load,
or if no cell prints at all, use `song_gpu_cell.py` from the site instead: one
cell to paste into a new notebook, which prints before it does anything else.
download `/song_gpu.ipynb` from the site and use **File → Upload notebook**.

The card goes away when the session ends — that is the trade for free. A T4 has
16 GB, twice the card the site was built on, so songs render faster while it runs.

In [ ]:
# Nothing above this line has to succeed for this to print. If it does not
# appear, no code is running: the runtime is not attached, and that is Colab
# rather than the notebook. Check the top-right of the page and Runtime ->
# Reconnect; a free T4 can also be out of capacity, in which case the CPU
# runtime still runs this, just slowly.
print('notebook is running; this cell printed, so the runtime is attached', flush=True)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess, sys
try:
    import torch
except ModuleNotFoundError:
    # Colab and Kaggle ship a CUDA build; a bare Linux box may have none. The
    # default Linux wheels bundle CUDA, so this is what puts the model on the
    # card rather than on the processor.
    print('torch is not installed on this runtime; installing the default CUDA build ...', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch', 'torchaudio', 'torchvision'], check=True)
    import torch
# A torch that is already here is left alone: swapping it for another wheel is
# how a working CUDA build turns into a CPU one.
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# The song service's own packages. Colab already ships a CUDA build of torch and
# torchaudio, so they are deliberately not named here: a pip wheel from PyPI would
# replace them with a CPU build and quietly move the model off the card.
print('1/4 system packages ...', flush=True)
!apt-get -qq install -y ffmpeg libsndfile1 git > /dev/null
print('2/4 the code ...', flush=True)
# The folder survives between runs inside one session, so it is updated rather
# than reused: code from an earlier run carries earlier pins, which is exactly how
# this card kept failing after the package list was fixed.
import os, subprocess, sys
REPO = '/content/internet'
GIT = 'https://github.com/soluzka/AI-driven_NVIDIA_song_designer.git'
if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '--depth', '1', 'origin', 'main'])
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'origin/main'])
    print('updated', REPO, flush=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', GIT, REPO])
print('code at commit',
      subprocess.run(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'],
                     capture_output=True, text=True).stdout.strip(), flush=True)
print('3/4 the packages (a few minutes, no output until pip finishes) ...', flush=True)
# The package list is song_service/requirements.txt, the same one the Windows
# installer uses, so every package ACE-Step imports is named (loguru, cutlet,
# fugashi, num2words, pypinyin, hangul-romanize, peft, accelerate, diffusers, ...)
# and the pins that matter come with it: transformers 4.57.6 with
# huggingface-hub 0.36.2, diffusers/gradio capped to what that hub allows,
# py3langid 0.2.2 for the segmenter.
import re, subprocess, sys
SKIP = ('torch', 'torchaudio', 'torchvision')
specs = []
for line in open('/content/internet/song_service/requirements.txt'):
    line = line.split('#')[0].strip()
    name = re.split(r'[=<>!\[ ]', line, 1)[0].strip().lower() if line else ''
    if name and name not in SKIP:
        specs.append(line)
print('installing', len(specs), 'packages ...', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + specs, check=True)
# ACE-Step's metadata pins transformers==4.50.0 against the 4.57.6 the lyric model
# needs, so the model's code is installed without its dependency list.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                'git+https://github.com/ace-step/ACE-Step.git'], check=True)
# The voice (chatterbox) is part of the work this card does: the song service calls
# it over loopback on 8095, as the Windows install runs it, so the tunnel keeps
# pointing at 8090 alone. --no-deps because its metadata pins torch 2.6 and
# numpy 1.x, which would replace this runtime's CUDA torch; everything it imports
# is listed in that file, so --no-deps loses nothing.
print('  the voice (chatterbox) ...', flush=True)
voice_specs = [line.split('#')[0].strip()
               for line in open('/content/internet/voice_service/requirements-shared.txt')
               if line.split('#')[0].strip()]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps'] + voice_specs,
               check=True)

print('4/4 checking the imports ...', flush=True)
import torch
try:
    from acestep.pipeline_ace_step import ACEStepPipeline
except ModuleNotFoundError as missing:
    print('missing package:', missing.name, '- install it with',
          sys.executable, '-m pip install', missing.name, flush=True)
    raise
print('packages in place; torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# The two checkpoints (about 10 GB, into the session disk): ACE-Step v1 3.5B for
# the music, Qwen2.5 1.5B for the words. install.py reads song_service/models.json
# and resumes rather than restarting a partial folder.
import os, subprocess
os.environ['HF_HOME'] = '/content/hf'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('downloading about 10 GB of weights; the bars below are the progress', flush=True)
result = subprocess.run(['python', 'song_service/install.py', '--weights'], cwd='/content/internet')
print('weights exit', result.returncode)
!du -sh /content/internet/song_service/models/* 2>/dev/null || true

In [ ]:
# --- repair a card that was set up before a fix ------------------------------
# Press this (not Run all) when a song fails with a package error, such as
# "type object 'LanguageIdentifier' has no attribute 'from_pickled_model'". It
# moves the code to main, pins the package ACE-Step calls, and stops the old
# service - that last step is the one a pip install cannot do for you, because a
# running process keeps the modules it imported. Then run the service cell below.
import importlib.metadata as metadata
import subprocess
import sys
import time


def run(*command):
    print('$', ' '.join(command), flush=True)
    answer = subprocess.run(command, capture_output=True, text=True)
    for stream in (answer.stdout, answer.stderr):
        if stream.strip():
            print(stream.strip()[:400], flush=True)
    return answer.returncode


run('git', '-C', '/content/internet', 'fetch', '--depth', '1', 'origin', 'main')
run('git', '-C', '/content/internet', 'reset', '--hard', 'origin/main')
run('git', '-C', '/content/internet', 'log', '-1', '--oneline')
try:
    print('py3langid before:', metadata.version('py3langid'), flush=True)
except Exception as exc:
    print('py3langid before: not installed -', exc, flush=True)
run(sys.executable, '-m', 'pip', 'install', '-q', 'py3langid==0.2.2')
print('py3langid after :', metadata.version('py3langid'), flush=True)
if run('pkill', '-f', 'app.py') == 0:
    print('stopped the service that was holding the old package', flush=True)
    time.sleep(2)
else:
    print('no service was running', flush=True)
print('now run the service cell below', flush=True)


In [ ]:
# The service on the notebook's own GPU, on every interface, warming the model now
# so the first visitor does not wait for the load. The wait prints, because a
# silent loop is indistinguishable from a hung one.
import os, subprocess, time, urllib.request
os.environ.update(SONG_PORT='8090', SONG_HOST='0.0.0.0', SONG_WARM='1', HF_HOME='/content/hf')
print('starting the service ...', flush=True)
# Re-running this cell after a package fix must replace the process that still
# holds the old modules in memory. The command line is "python app.py", so the
# match is on that, not on a path that never appears in it.
import sys
if subprocess.run(['pkill', '-f', 'app.py'], capture_output=True).returncode == 0:
    print('stopped the services from the previous run so the installed packages load', flush=True)
    time.sleep(2)
langid_version = subprocess.run(
    [sys.executable, '-c', 'import importlib.metadata as m; print(m.version("py3langid"))'],
    capture_output=True, text=True).stdout.strip()
print('py3langid', langid_version or 'unknown', '(ACE-Step needs 0.2.2)', flush=True)
log = open('/content/song.log', 'w')
proc = subprocess.Popen([sys.executable, 'app.py', '--role', 'song'], cwd='/content/internet/song_service',
                        stdout=log, stderr=subprocess.STDOUT)
health = ''
for attempt in range(36):
    try:
        health = urllib.request.urlopen('http://127.0.0.1:8090/health', timeout=5).read().decode()
        break
    except Exception:
        if attempt and attempt % 6 == 0:
            print('  waiting for /health ... ' + str(attempt * 5) + 's', flush=True)
        time.sleep(5)
if health:
    print('service:', health[:300])
else:
    print('the service did not answer within 3 minutes. Last lines of /content/song.log:')
    print()
    print(open('/content/song.log').read()[-2000:])
    print()
    print('Rerun this cell after reading that: the usual causes are a package that did')
    print('not install in the cell above, or the GPU runtime not being selected.')

# --- the voice service ------------------------------------------------------
# The song service sends the words and the reference clip to 8095 over loopback,
# so nothing extra is exposed publicly and no second tunnel is needed.
import sys
print('starting the voice service on 8095 ...', flush=True)
os.environ['VOICE_PORT'] = '8095'
voice_log = open('/content/voice.log', 'w')
voice_proc = subprocess.Popen([sys.executable, 'app.py'],
                              cwd='/content/internet/voice_service',
                              stdout=voice_log, stderr=subprocess.STDOUT)
voice_health = ''
for attempt in range(24):
    try:
        voice_health = urllib.request.urlopen('http://127.0.0.1:8095/health', timeout=5).read().decode()
        break
    except Exception:
        if attempt and attempt % 4 == 0:
            print('  waiting for the voice ... ' + str(attempt * 5) + 's', flush=True)
        time.sleep(5)
if voice_health:
    print('voice:', voice_health[:200], flush=True)
else:
    print('the voice service did not answer within 2 minutes. Last lines of /content/voice.log:')
    print(open('/content/voice.log').read()[-1500:])


In [ ]:
# --- the video downloader, on this box, for phones -------------------------
# A phone cannot download a video by itself: the site's button asks 127.0.0.1:8080,
# which on a phone is the phone, and the Worker's own proxy is switched off. The
# song service forwards /pc/<path> to 127.0.0.1:8080, so starting downloader_service
# here makes this card serve downloads - and Top 100 and the player - through the
# same tunnel the songs already use. No second tunnel, no extra address.
import os, subprocess, sys, time, urllib.request

print('the video downloader on 8080 ...', flush=True)
# Its own packages only. None of them touch torch, so the CUDA build stays as it is.
if subprocess.run([sys.executable, '-c', 'import yt_dlp, flask, flask_cors, imageio_ffmpeg'],
                  capture_output=True).returncode != 0:
    print('  installing its packages (about a minute) ...', flush=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'Flask>=3.1.0', 'Flask-Cors>=6.0.0', 'yt-dlp>=2026.8.0',
                    'curl_cffi>=0.11,<0.13', 'imageio-ffmpeg>=0.6.0',
                    'requests>=2.31.0'], check=True)

# A rerun has to replace the process that still holds 8080 and the old code. Its
# command line is "python app.py", the same as the song service's, so it is killed
# by the pid this cell wrote last time rather than by a name that would match both.
PIDFILE = '/content/downloads.pid'
try:
    old = int(open(PIDFILE).read().strip())
    os.kill(old, 15)
    print('  stopped the downloader from the previous run (pid ' + str(old) + ')', flush=True)
    time.sleep(1)
except Exception:
    pass

download_log = open('/content/downloads.log', 'w')
download_env = dict(os.environ, PORT='8080', MAX_HEIGHT='720')
download_proc = subprocess.Popen([sys.executable, 'app.py'],
                                 cwd='/content/internet/downloader_service',
                                 stdout=download_log, stderr=subprocess.STDOUT,
                                 env=download_env)
try:
    open(PIDFILE, 'w').write(str(download_proc.pid))
except OSError:
    pass

download_health = ''
for attempt in range(15):
    try:
        download_health = urllib.request.urlopen('http://127.0.0.1:8080/health', timeout=5).read().decode()
        break
    except Exception:
        if attempt and attempt % 5 == 0:
            print('  waiting for the downloader ... ' + str(attempt * 2) + 's', flush=True)
        time.sleep(2)
if download_health:
    print('downloads:', download_health[:220], flush=True)
    print('  a phone now reaches it at <this card address>/pc/api/download', flush=True)
else:
    print('the downloader did not answer on 8080 within 30s. Last lines of /content/downloads.log:')
    try:
        print(open('/content/downloads.log').read()[-1200:])
    except Exception as exc:
        print('no log at all:', exc)
    print()
    print('Songs still work without it; only downloads and Top 100 are missing.')


In [ ]:
# Optional. The join key puts this card in the pool, so every visitor's click can use
# it. It is looked for in this order, first one wins:
#   1. between the quotes below (a copy of this notebook, not the public one);
#   2. SONG_JOIN_KEY in the environment;
#   3. the Colab secret named SONG_JOIN_KEY (key icon, left edge): one paste there and
#      every run of this notebook joins the pool by itself - no editing this cell.
# With none of them the next cell prints an address for Connect my model instead.
#
# Both routes join the same pool - this notebook, and the Kaggle kernel the site's
# Start Kaggle pushes - and the key is the only door. This notebook is public, so
# anyone may run it; a card without SONG_JOIN_KEY in that person's own Colab secrets
# never joins your pool, and the address it prints is theirs alone.
#
# This is a cell and not a prompt on purpose: getpass waits for a terminal the
# notebook does not give it, so the cell sat loading forever and no address was
# ever printed.
JOIN_KEY = ''

import os


def join_key():
    """The join key for this session, or '' to stay private.

    The key cannot be shipped inside this notebook: the file is public, and the key is
    the only thing standing between the pool and anyone's GPU. Reading it from a Colab
    secret is what makes joining automatic after one paste.
    """
    typed = (globals().get('JOIN_KEY') or '').strip()
    if typed:
        print('join key: from JOIN_KEY in this cell')
        return typed
    from_env = (os.environ.get('SONG_JOIN_KEY') or '').strip()
    if from_env:
        print('join key: from SONG_JOIN_KEY in the environment')
        return from_env
    try:
        from google.colab import userdata
        secret = (userdata.get('SONG_JOIN_KEY') or '').strip()
        if secret:
            print('join key: from the Colab secret SONG_JOIN_KEY')
            return secret
    except Exception:
        pass
    return ''


In [ ]:
# The tunnel, and one unmistakable block with the address to paste.
#
# A trycloudflare name dies with the process that made it, but /content survives
# between runs inside one session: the old name is still sitting in the log after
# cloudflared is gone. That is why a reused address gets pasted and never turns
# green - the site asks /health and nothing answers. So a saved name is only
# reused when it answers, and a new tunnel always starts from an empty log.
#
# No "!" lines in here on purpose. A line starting with "!" is a Colab feature, not
# Python, and one placed inside an if/else block is exactly the kind of thing that
# turns a cell into a syntax error instead of output: plain Python below, nothing
# for the notebook layer to translate.
import os, re, time, json, urllib.request, subprocess

try:
    JOIN_KEY = join_key()
except NameError:  # this cell run on its own, without the key cell above
    JOIN_KEY = globals().get('JOIN_KEY', '').strip()
SITE = 'https://browser.bpiersing.workers.dev'
BINARY = '/usr/local/bin/cloudflared'
URL = 'https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64'
LOG = '/content/tunnel.log'


def show(url, in_pool=False):
    print()
    print('=' * 74)
    if in_pool:
        print('  THIS CARD IS IN THE POOL')
        print()
        print('  Nothing to paste: ' + SITE + ' finds it by itself, and every visitor')
        print('  can use it. The header names it as a pooled card within a minute.')
    else:
        print('  COPY THIS ADDRESS')
        print()
        print('  ' + url)
        print()
        print('  Then on the site: ' + SITE)
        print('  Press Connect my model in the header, paste it into the box, press OK.')
        print('  The dot beside the button turns green when that machine answers.')
    print('=' * 74)
    try:
        open('/content/ADDRESS.txt', 'w').write(url)
        print('  (also saved to /content/ADDRESS.txt)')
    except Exception:
        pass


def answers(url, timeout=8):
    """True when the model service behind that address replies to /health."""
    try:
        with urllib.request.urlopen(url.rstrip('/') + '/health', timeout=timeout) as reply:
            return reply.status == 200
    except Exception:
        return False


def logged_url():
    """The last address written to the tunnel log, answered or not."""
    try:
        found = re.findall(r'https://[a-z0-9-]+[.]trycloudflare[.]com', open(LOG).read())
    except FileNotFoundError:
        return ''
    return found[-1] if found else ''


url = ''
saved = logged_url()

if saved and answers(saved):
    url = saved
    print('the tunnel already running in this session answers, so there is nothing to wait for')
else:
    if saved:
        print(saved, 'is the last address in the log, and it does not answer any more.')
        print('Starting a new tunnel: a trycloudflare name stops when its process stops.')
    print('1/2 downloading cloudflared (40 MB, progress below) ...', flush=True)
    with urllib.request.urlopen(URL, timeout=120) as answer, open(BINARY, 'wb') as out:
        total = int(answer.headers.get('content-length') or 0)
        got = 0
        next_mark = 8 * 1024 * 1024
        while True:
            chunk = answer.read(256 * 1024)
            if not chunk:
                break
            out.write(chunk)
            got += len(chunk)
            if got >= next_mark:
                print('  %.0f MB' % (got / 1048576.0), flush=True)
                next_mark += 8 * 1024 * 1024
    os.chmod(BINARY, 0o755)
    print('2/2 starting the tunnel ...', flush=True)
    # Empty first, so the name read below can only come from this process.
    try:
        open(LOG, 'w').close()
    except OSError:
        pass
    subprocess.Popen([BINARY, 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8090',
                      '--logfile', LOG], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for attempt in range(30):
        url = logged_url()
        if url:
            break
        if attempt and attempt % 5 == 0:
            print('  still waiting for a name ... ' + str(attempt * 2) + 's', flush=True)
        time.sleep(2)
    if url:
        # A brand new name takes a few seconds to route, so it is probed before
        # being handed over: printing an address that does not answer is how this
        # went wrong before.
        for attempt in range(12):
            if answers(url):
                break
            time.sleep(5)

if url and answers(url):
    # The card announces itself before any address is printed: a pooled card needs no
    # address in the page at all, and telling someone to paste one when the site
    # already knows the machine is the paste this cell exists to remove.
    joined = False
    if JOIN_KEY:
        body = json.dumps({'url': url, 'note': 'colab t4'}).encode()
        request = urllib.request.Request('https://song-pool.bpiersing.workers.dev/join', data=body,
                                         headers={'content-type': 'application/json', 'x-join-key': JOIN_KEY})
        try:
            print('pool:', urllib.request.urlopen(request, timeout=20).read().decode())
            joined = True
        except Exception as exc:
            print('join refused:', exc)
    show(url, joined)
    if joined:
        print('  Keep this tab open: the card leaves the pool when the session ends.')
    else:
        print('  To have the site find it by itself next time, add SONG_JOIN_KEY to')
        print('  Colab Secrets (key icon, left): one paste there, and every run of this')
        print('  notebook joins the pool with nothing to paste here.')
elif url:
    print()
    print('The tunnel got the name ' + url + ' but /health does not answer through it.')
    print('So it is not offered as an address: the site would only show a red dot.')
    print('Check that the cell above printed "service:" with a GPU in it, then rerun this cell.')
else:
    print('cloudflared did not produce a name. Last lines of ' + LOG + ':')
    try:
        print(open(LOG).read()[-1500:])
    except Exception as exc:
        print('no log at all:', exc)
        print('That means the download above failed; rerun this cell.')
